# Light EDA + Fixed Split Audit

Notebook nay dung de kiem tra nhanh data Agriculture va audit split co dinh trong repo.

- `train/` co label trong ten file, dung de train va validation noi bo.
- `val/` cua Kaggle khong co label, chi dung de tao `submission.csv`.
- Split mac dinh doc tu `splits/seed42_val20/split_manifest.csv` de moi lan thu nghiem cong bang.


In [ ]:
from pathlib import Path
from collections import Counter
import os
import random

import pandas as pd
import numpy as np

try:
    from PIL import Image
except Exception as exc:
    Image = None
    print('PIL import failed:', exc)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print('matplotlib import failed:', exc)

VAL_SPLIT = 0.2
SEED = 42
SPLIT_MANIFEST_PATH = Path('splits/seed42_val20/split_manifest.csv')

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'run.sh').exists():
            return p
    return start

REPO_ROOT = find_repo_root()

def find_data_dir():
    candidates = []
    env_data = os.environ.get('DATA_DIR')
    if env_data:
        candidates.append(Path(env_data))
    candidates.extend([
        Path('/kaggle/input/datasets/nadkli/data-agriculture/dataset'),
        Path('/kaggle/input/data-agriculture/dataset'),
        REPO_ROOT / 'data',
        REPO_ROOT / 'dataset',
    ])
    for path in candidates:
        if (path / 'train' / 'RGB').exists() and (path / 'val' / 'RGB').exists():
            return path.resolve()
    raise FileNotFoundError('Khong tim thay data dir. Hay set os.environ["DATA_DIR"] truoc khi chay notebook.')

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else REPO_ROOT / 'outputs' / 'eda_light_split'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT :', REPO_ROOT)
print('DATA_DIR  :', DATA_DIR)
FIXED_SPLIT_PATH = (REPO_ROOT / SPLIT_MANIFEST_PATH).resolve()
print('OUTPUT_DIR:', OUTPUT_DIR)
print('FIXED_SPLIT_PATH:', FIXED_SPLIT_PATH)


In [ ]:
def label_from_filename(fname):
    return Path(fname).name.split('_')[0]

def base_name(path):
    return Path(path).stem

def list_files(split, modality):
    folder = DATA_DIR / split / modality
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file()])

records = []
for split in ['train', 'val']:
    for modality in ['RGB', 'HS', 'MS']:
        for p in list_files(split, modality):
            records.append({
                'split': split,
                'modality': modality,
                'filename': p.name,
                'base_name': p.stem,
                'extension': p.suffix.lower(),
                'size_bytes': p.stat().st_size,
                'label': label_from_filename(p.name) if split == 'train' else None,
                'path': str(p),
            })

df_files = pd.DataFrame(records)
print('Total files:', len(df_files))
display(df_files.groupby(['split', 'modality', 'extension']).size().rename('count').reset_index())


In [ ]:
train_rgb = df_files[(df_files['split'] == 'train') & (df_files['modality'] == 'RGB')].copy()
class_counts = train_rgb['label'].value_counts().sort_index()
display(class_counts.rename_axis('class_name').reset_index(name='count'))

if plt is not None:
    ax = class_counts.plot(kind='bar', figsize=(6, 3), rot=0, title='Train class distribution')
    ax.set_xlabel('class')
    ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()


In [ ]:
def modality_alignment(split):
    sets = {}
    for modality in ['RGB', 'HS', 'MS']:
        part = df_files[(df_files['split'] == split) & (df_files['modality'] == modality)]
        sets[modality] = set(part['base_name'])
    union = set().union(*sets.values()) if sets else set()
    rows = []
    for modality in ['RGB', 'HS', 'MS']:
        missing = sorted(union - sets[modality])
        rows.append({
            'split': split,
            'modality': modality,
            'count': len(sets[modality]),
            'missing_count': len(missing),
            'missing_examples': ', '.join(missing[:5]),
        })
    return pd.DataFrame(rows)

alignment_df = pd.concat([modality_alignment('train'), modality_alignment('val')], ignore_index=True)
display(alignment_df)
alignment_df.to_csv(OUTPUT_DIR / 'modality_alignment.csv', index=False)


In [ ]:
rgb_df = df_files[df_files['modality'] == 'RGB'].copy()
size_summary = rgb_df.groupby('split')['size_bytes'].describe().reset_index()
display(size_summary)

small_rgb = rgb_df[rgb_df['size_bytes'] < 200].sort_values(['split', 'filename'])
print('Small RGB files (<200 bytes):', len(small_rgb))
display(small_rgb[['split', 'filename', 'size_bytes', 'path']].head(30))
small_rgb.to_csv(OUTPUT_DIR / 'small_rgb_files.csv', index=False)


In [ ]:
def inspect_rgb_images(paths, max_files=200):
    rows = []
    if Image is None:
        return pd.DataFrame(rows)
    for p in list(paths)[:max_files]:
        try:
            with Image.open(p) as img:
                rows.append({
                    'filename': p.name,
                    'width': img.size[0],
                    'height': img.size[1],
                    'mode': img.mode,
                    'ok': True,
                    'error': '',
                })
        except Exception as exc:
            rows.append({
                'filename': p.name,
                'width': None,
                'height': None,
                'mode': None,
                'ok': False,
                'error': str(exc),
            })
    return pd.DataFrame(rows)

rgb_paths = [Path(p) for p in rgb_df['path'].tolist()]
rgb_inspect = inspect_rgb_images(rgb_paths, max_files=len(rgb_paths))
display(rgb_inspect.groupby(['ok', 'width', 'height', 'mode'], dropna=False).size().rename('count').reset_index().head(20))
display(rgb_inspect[~rgb_inspect['ok']].head(20))
rgb_inspect.to_csv(OUTPUT_DIR / 'rgb_image_inspection.csv', index=False)


In [ ]:
def inspect_tiff_metadata(split, modality, max_files=5):
    rows = []
    if Image is None:
        return pd.DataFrame(rows)
    paths = list_files(split, modality)[:max_files]
    for p in paths:
        try:
            with Image.open(p) as img:
                rows.append({
                    'split': split,
                    'modality': modality,
                    'filename': p.name,
                    'size': img.size,
                    'mode': img.mode,
                    'bands': img.getbands(),
                    'n_frames': getattr(img, 'n_frames', 1),
                    'size_bytes': p.stat().st_size,
                })
        except Exception as exc:
            rows.append({'split': split, 'modality': modality, 'filename': p.name, 'error': str(exc)})
    return pd.DataFrame(rows)

spectral_meta = pd.concat([
    inspect_tiff_metadata('train', 'HS'),
    inspect_tiff_metadata('train', 'MS'),
    inspect_tiff_metadata('val', 'HS'),
    inspect_tiff_metadata('val', 'MS'),
], ignore_index=True)
display(spectral_meta)
spectral_meta.to_csv(OUTPUT_DIR / 'spectral_tiff_metadata_sample.csv', index=False)


In [ ]:
manifest_path = FIXED_SPLIT_PATH
if not manifest_path.exists():
    raise FileNotFoundError(f'Khong tim thay fixed split manifest: {manifest_path}')

split_df = pd.read_csv(manifest_path)
required_cols = {'filename', 'label', 'split'}
missing_cols = required_cols - set(split_df.columns)
if missing_cols:
    raise ValueError(f'Split manifest thieu cot: {sorted(missing_cols)}')

split_df['split'] = split_df['split'].str.lower()
if not set(split_df['split']).issubset({'train', 'val'}):
    raise ValueError('Cot split chi duoc gom train/val')
if split_df['filename'].duplicated().any():
    display(split_df[split_df['filename'].duplicated(keep=False)].sort_values('filename'))
    raise ValueError('Split manifest bi trung filename')

available_train_rgb = set(train_rgb['filename'].tolist())
manifest_files = set(split_df['filename'].tolist())
missing_in_data = sorted(manifest_files - available_train_rgb)
extra_in_data = sorted(available_train_rgb - manifest_files)

print('Fixed split manifest:', manifest_path)
print('Rows in manifest:', len(split_df))
print('Missing manifest files in DATA_DIR/train/RGB:', len(missing_in_data))
print('Extra DATA_DIR/train/RGB files not in manifest:', len(extra_in_data))

if missing_in_data:
    display(pd.DataFrame({'missing_in_data': missing_in_data[:30]}))
if extra_in_data:
    display(pd.DataFrame({'extra_in_data': extra_in_data[:30]}))

split_summary = (
    split_df.groupby(['label', 'split']).size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['train', 'val']:
    if col not in split_summary.columns:
        split_summary[col] = 0
split_summary['total'] = split_summary['train'] + split_summary['val']
train_total = int(split_summary['train'].sum())
val_total = int(split_summary['val'].sum())
split_summary['val_ratio'] = split_summary['val'] / split_summary['total']
split_summary = split_summary[['label', 'total', 'train', 'val', 'val_ratio']]

display(split_summary)
print(f'Total fixed split: train={train_total}, val={val_total}')

manifest_copy_path = OUTPUT_DIR / 'fixed_split_manifest_checked.csv'
summary_path = OUTPUT_DIR / 'fixed_split_summary_checked.csv'
split_df.to_csv(manifest_copy_path, index=False)
split_summary.to_csv(summary_path, index=False)
print('Saved:', manifest_copy_path)
print('Saved:', summary_path)


In [ ]:
if Image is not None and plt is not None:
    samples = []
    for label, group in train_rgb.groupby('label'):
        samples.extend(group.sort_values('filename').head(3)['path'].tolist())
    n = len(samples)
    if n:
        cols = 3
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
        axes = np.array(axes).reshape(-1)
        for ax, path_str in zip(axes, samples):
            p = Path(path_str)
            try:
                with Image.open(p) as img:
                    ax.imshow(img.convert('RGB'))
                ax.set_title(f'{label_from_filename(p.name)}\n{p.name}', fontsize=8)
            except Exception as exc:
                ax.set_title(f'ERR: {p.name}\n{exc}', fontsize=8)
            ax.axis('off')
        for ax in axes[n:]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('Skip preview because PIL or matplotlib is unavailable.')
